---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-35: LangChain <b>Text Splitters (Chunking)</b></h1>

# Learning agenda of this notebook  

1. Recap of Core Components of LangChain
2. Recap of LangChain Indexes Component
3. Text Splitters in LangChain
    - Document Chunking
    - Chunking Types
    - Overlapping Chunks
4. Hands-on Practice of Different Chunking Techniques
    - Example 1: `CharacterTextSplitter`
    - Example 2: `TokenTextSplitter`
    - Example 3: `RecursiveCharacterTextSplitter`
    - Example 4: Code-Aware Text Splitting using `RecursiveCharacterTextSplitter` with `Language` enum
    - Example 5: `SemanticChunker`
    - Example 6: `AgenticChunker`

# <span style='background :lightgreen' >1. Recap: Core Components of [LangChain](https://github.com/langchain-ai/langchain)</span>


<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">LangChain is an open source framework that provides us with a set of tools and abstractions that make it easier for us to create complex LLM-powered applications.</div></h3>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Langchain components can be chained together to create sophisticated AI applications like chatbots, question-answering systems, and intelligent agents.</div></h3>


| Component | Description | Key Benefit |
|-----------|--------------|-------------|
| **Models** | LangChain's "universal remote control" for AI—one interface to rule them all, freeing developers from vendor-specific complexity and enabling true AI provider independence. | Write once, use with any AI model |
| **Prompts** | Reusable, parameterized templates that standardize how you communicate with AI models. | Consistent messaging with easy updates | 
| **Memory** | LangChain Memory enables AI models to maintain context across multiple interactions in a conversation. Without memory, each interaction is independent - the model has no knowledge of previous exchanges. | Natural, flowing conversations |
| **Chains** | LangChain’s “AI Assembly Lines” — they connect multiple models, prompts, or tools together into a logical flow of reasoning, automating complex multi-step tasks with simplicity and consistency. | Transform complex tasks into simple sequences | 
| **Indexes** | Indexes are LangChain's smart knowledge management systems that convert your raw documents (PDFs, websites, databases) into searchable libraries where AI models can quickly find and retrieve only the most relevant information w/o exceeding context limits. | Fast retrieval, reduced API costs | 
| **Agents** | LangChain Agents are the “decision-making brains” of your AI system — they enable models to reason, plan, and dynamically decide which tools to use and what actions to take in order to achieve a user’s goal.| Autonomous problem-solving without manual programming |

# <span style='background :lightgreen' >2. Recap of LangChain Indexes Component</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">LangChain Indexes consist of four interconnected components that work together to enable Retrieval Augmented Generation (RAG).</div></h3>

<img align="right" width="800" src="../images/r3.png"  >

- [**Document Loaders**](https://docs.langchain.com/oss/javascript/integrations/document_loaders) provide a standard interface (160+ integrations) for reading data from different sources ike PDFs, web pages, Google Drive, Slack, Notion, and more into LangChain’s Document format. This ensures that data can be handled consistently regardless of the source. Common categories include:
    - **TextLoaders**  for plain .txt files (`CSVLoader`, `Docx2txtLoader`, `JSONLoader`, `PyPDFLoader`, `MarkdownLoader` etc)
    - **Web loaders** for ingesting data from online or cloud-based sources. (`WebBaseLoader`, `SitemapLoader`, `S3FileLoader`, `GCSFileLoader`, etc)
    - **Productivity tools** for pulling data from collaboration and enterprise platforms. (`GoogleDriveLoader`, `YouTubeLoader`, `TwitterLoader`, `WikipediaLoader`, `NotionPageLoader`, `ConfluenceLoader` etc)
- [**Text Splitters**](https://docs.langchain.com/oss/javascript/integrations/splitters/index#text-splitters) break large docs into smaller chunks that will be retrievable individually, ensuring that each chunk contains meaningful information while staying within model context window limit. Main strategies are:
    - **Text structure-based:** Splits hierarchically (paragraphs → sentences → words)
    - **Length-based:** Splits by token or character count
    - **Document structure-based:** Splits based on format (HTML, Markdown, code)
- [**Vector Stores**](https://docs.langchain.com/oss/python/integrations/vectorstores/index) are specialized databases for storing and searching embeddings. They store embedded documents and perform similarity search to find semantically similar content. LangChain provides a unified interface with methods like addDocuments, delete, and similaritySearch. Popular options include Chroma, Pinecone, Qdrant, FAISS, MongoDB Atlas, and in-memory stores for testing.
- [**Retrievers**](https://docs.langchain.com/oss/python/integrations/retrievers/index#retrievers) accepts an unstructured query as input, search through the vector database to find the most relevant document chunks and return a list of Documents as output.

# <span style='background :lightgreen' >3. Text Splitters in LangChain</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Text Splitting (chunking) is the process of breaking large chunks of text (like articles, PDFs, HTML pages, or books) into smaller, manageable pieces (chunks) that an LLM can handle effectively.</div></h3>

## a. Document Chunking
- Chunking is the process of breaking down large pieces of text into smaller segments called chunks. It's an essential preprocessing technique that helps optimize the relevance of the content stored in a vector database.
- When working with large documents — such as books, reports, or research papers — we can't feed the entire text into an embedding model at once.
- All embedding models have context windows, which determine the amount of information in tokens that can be processed into a single fixed-size vector. Exceeding this context window may result in truncation or throwing away excess tokens before being processed into a vector, which is potentially harmful as important context could be removed from the representation.
- Even if embedding models didn't have limits, long texts would produce blurry embeddings that mix multiple topics together, reducing retrieval accuracy.
- A business document may have dozens or hundreds (or more) vectors, each containing a small part of the original document.


<p align="center">
  <img src="../images/t4.png" alt="Complex Chain Flow" style="width: 85%;">
</p>

### Key Benefits of Chunking
- **Improved Accuracy:**
    - Smaller chunks create more precise embeddings
    - Semantic search returns highly relevant results
    - Reduces information overload for the model
- **Better Performance:**
    - Faster processing of smaller text segments
    - More efficient memory usage
    - Better parallelization opportunities
- **Reduced Hallucination:**
    - Prevents models from losing track of context
    - Maintains topic coherence in summaries
    - Reduces drift in long-form content processing
- **Enhanced Search Quality:**
    - Returns specific, relevant information
    - Eliminates noise from unrelated content
    - Improves retrieval precision

### Optimal Chunk Size Guidelines
- Based on recent research, recursive character text splitting with appropriate chunk sizes (200–400 tokens) and minimal overlap offers a good balance between simplicity and effectiveness for most applications.
- **Key Recommendations:**
    - **Chunk Size:** Chunk size should be 200–500 tokens for most use cases
    - **Overlap:** We recommend starting with an overlap of approximately 10%. For example, given a fixed chunk size of 256 tokens, you would begin testing with an overlap of 25 tokens
    - **Context Window:** Consider the maximum context window of your LLM (e.g. 4,096 tokens for GPT 3.5 Turbo)
    - **Embedding Model Limits:** Optimize for your vector database - ChromaDB / FAISS: Chunk size should align with embedding model limits (e.g. OpenAI's text-embedding-ada-002 supports 8192 tokens)

## b. Chunking Types
- Evaluating Chunking Strategies for Retrieval: https://www.trychroma.com/research/evaluating-chunking


| **Chunking Type** | **LangChain Class** | **Description** | **How It Works** | **Example Use Case** | **Pros & Cons** |
|---|---|---|---|---|---|
| **Fixed-Size (Character)** | `CharacterTextSplitter` | Splits text every fixed number of characters using a single separator | Split document every 1,000 characters with overlap | Simple method for uniform data like articles or notes | ✅ Simple, predictable<br>❌ May cut mid-sentence |
| **Fixed-Size (Token/OpenAI)** | `TokenTextSplitter` | Splits based on token count using OpenAI's `tiktoken` tokenizer | Split every 256 tokens with 25-token overlap | Homogeneous content when OpenAI model token limits matter | ✅ Respects model limits<br>❌ Needs `tiktoken` installed |
| **Fixed-Size (Token/HuggingFace)** | `SentenceTransformersTokenTextSplitter` | Splits by token count using HuggingFace's sentence-transformers tokenizer | Split every 256 tokens aligned to embedding model vocabulary | Preparing chunks for HuggingFace embedding models | ✅ Embedding-model aware<br>❌ Needs `sentence-transformers` |
| **Sentence-Based (spaCy)** | `SpacyTextSplitter` | Uses spaCy NLP to detect linguistically accurate sentence boundaries | Identify sentence boundaries via spaCy's NLP pipeline | Q&A systems, summarization tasks needing accurate sentences | ✅ Linguistically accurate<br>❌ Needs `spacy` + language model |
| **Sentence-Based (NLTK)** | `NLTKTextSplitter` | Uses NLTK's sentence tokenizer to split on sentence boundaries | Splits text using NLTK's `sent_tokenize()` | Sentence-level retrieval on general English text | ✅ Lightweight, no GPU needed<br>❌ Less accurate than spaCy |
| **Paragraph-Based** | `CharacterTextSplitter(separator="\n\n")` | Splits at paragraph boundaries using double newlines as the separator | Uses `"\n\n"` as the natural paragraph boundary delimiter | Cleanly formatted text (reports, blogs, academic papers) | ✅ Natural structure<br>❌ Inconsistent chunk sizes |
| **Recursive Character** | `RecursiveCharacterTextSplitter` | Splits recursively using a hierarchy of separators until chunks are small enough | Tries `"\n\n"` → `"\n"` → `" "` → `""` in order | General-purpose; works across all document types | ✅ Adaptive, most recommended<br>❌ Moderately complex |
| **Markdown-Aware** | `MarkdownTextSplitter` | `RecursiveCharacterTextSplitter` pre-configured with Markdown separators | Splits on `##`, `###`, ` ``` `, `---` to preserve doc structure | Markdown docs, README files, wikis, Jupyter notebooks | ✅ Preserves heading hierarchy<br>❌ Only works on Markdown |
| **Code-Aware (Python)** | `PythonCodeTextSplitter` | `RecursiveCharacterTextSplitter` pre-configured with Python syntax separators | Splits on `class`, `def`, `\n` to keep code blocks intact | Python source files, code documentation | ✅ Keeps functions/classes intact<br>❌ Python-only |
| **Code-Aware (Multi-language)** | `RecursiveCharacterTextSplitter.from_language(Language.X)` | Language-aware splitter supporting JS, TS, Go, Java, C, C++, Ruby, Rust, Scala, Swift, Markdown, LaTeX, HTML | Selects language-specific separators automatically | Splitting source code files in any supported language | ✅ Supports 15+ languages<br>❌ No semantic understanding |
| **LaTeX-Aware** | `LatexTextSplitter` | `RecursiveCharacterTextSplitter` pre-configured for LaTeX document structure | Splits on `\section`, `\subsection`, `\begin{}` etc. | Academic papers, scientific documents written in LaTeX | ✅ Preserves LaTeX structure<br>❌ Only works on LaTeX |
| **HTML-Aware** | `HTMLHeaderTextSplitter` | Splits HTML documents based on header tags and attaches header metadata to chunks | Splits on `<h1>`, `<h2>`, `<h3>` and preserves header hierarchy in metadata | Web pages, HTML documentation, scraped content | ✅ Retains page structure<br>❌ Requires well-formed HTML |
| **Semantic** | `SemanticChunker` *(langchain_experimental)* | Uses embedding models to detect natural topic breaks and split accordingly | Groups semantically similar sentences using cosine distance on embeddings | RAG apps, chatbots, complex narratives with topic shifts | ✅ Best semantic quality<br>❌ Computationally expensive |
| **LLM-Aware** | Custom via `LLMChain` + `RunnableLambda` | Uses an LLM to intelligently determine split boundaries based on content understanding | Prompts an LLM to identify logical chunk boundaries in the document | Complex documents requiring deep semantic understanding | ✅ Most accurate<br>❌ High cost, slow |
| **Contextual Retrieval** | Custom via Anthropic Claude API | Prompts Claude with full document + chunk to generate a contextual description prepended to each chunk before embedding | Each chunk is enriched with a generated summary of its role in the larger document | Production RAG systems needing better context preservation | ✅ Best retrieval accuracy<br>❌ Requires prompt caching for cost control |

---

### Decision Tree: Which Chunking Strategy to Use?

```
Do you have large documents?
│
├─ YES → What type of content is it?
│         ├─ SOURCE CODE  → Use Code-Aware Splitter (PythonCodeTextSplitter / Language.X)
│         ├─ MARKDOWN     → Use MarkdownTextSplitter
│         ├─ LATEX        → Use LatexTextSplitter
│         ├─ HTML         → Use HTMLHeaderTextSplitter
│         └─ GENERAL TEXT → Is topic coherence important?
│                           ├─ YES (cost is ok)    → Use SemanticChunker
│                           ├─ YES (budget limit)  → Use RecursiveCharacterTextSplitter
│                           ├─ NEED EXACT TOKENS   → Use TokenTextSplitter (OpenAI)
│                           │                         or SentenceTransformersTokenTextSplitter (HuggingFace)
│                           └─ NEED SENTENCES      → Use SpacyTextSplitter / NLTKTextSplitter
│
└─ NO → Consider not chunking if content fits the context window
         └─ If near limit → Use RecursiveCharacterTextSplitter with large chunk_size
```
## Installation Required
```
$ uv add langchain-text-splitters
$ uv add spacy nltk
$ uv add langchain-experimental langchain-huggingface sentence-transformers
```

## c. Overlapping Chunks
### Chunk overlap means how many words You want to overlap
- Chunk overlap is the number of characters or tokens that are shared between consecutive chunks to maintain context continuity. For example, if chunk 1 ends with "exploring Mars" and has 20-character overlap, chunk 2 might start with "Mars, humanity continues" - preserving the connection between chunks. This prevents important information from being lost at chunk boundaries but increases total processing overhead.
- More chunk Overlap = More No of Chunks
- According to research 20% of text are good size for chunk overlap

### Overlapping Chunks
- Sometimes splitting text into chunks can cut through a sentence or idea, causing loss of context between chunks.
- Chunk overlap is the number of characters or tokens that are shared between consecutive chunks to maintain context continuity preventing important information from being lost at chunk boundaries.
- Example with Overlap: If you split every 100 words with 20-word overlap:
```
Chunk 1 → Words 1–100
Chunk 2 → Words 81–180 (words 81–100 overlap with Chunk 1)
Chunk 3 → Words 161–260 (words 161–180 overlap with Chunk 2)
```
- **Overlap Considerations:**
    - **High Overlap (20-30%):** Improves recall but increases storage and compute costs
    - **Low Overlap (5-10%):** Saves space but may fragment information
    - **Sweet Spot:** 10-15% overlap works well for many scenarios

In [1]:
# Import the appropriate loader
from langchain_community.document_loaders import TextLoader

loader = TextLoader('../data/arif_bio.txt', encoding='utf-8')   # Create loader instance by passing it the file  path
docs = loader.load()                                            # Load the document
print(f"Loaded {len(docs)} document")                           # Text loaders consider whole text file as one chunk  
text = docs[0].page_content
print(f"File Content:\n {text}")                                # Will be passed to text splitter

Loaded 1 document
File Content:
 Dr. Muhammad Arif Butt is an accomplished Assistant Professor at the Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. He holds an MSc and MPhil (both with Gold Medals) and a Ph.D. in Computer Science from PUCIT, University of the Punjab. His research focuses on fuzzy inference models applied to operating systems, embedded systems, and cloud-based services, particularly in decision-making under uncertain and imprecise conditions.

With over 33 years of experience in teaching and management, Dr. Butt has served in both the Pakistan Army and University of the Punjab, bringing a wealth of interdisciplinary expertise. His teaching specializations include embedded and real-time operating systems, system programming, cybersecurity, and artificial intelligence.

Beyond academia, he is a technology entrepreneur, serving as the Founder of Excaliat and Falcon-Hunt and Co-Founder of Tbox Solutionz. In recent years, he has gained signific

# <span style='background :lightgreen' >4. Hands-on Practice of Different Chunking Techniques</span>

- Use following Online Chunking Tools for better understanding:
    - **ChunkViz:**  https://chunkviz.up.railway.app/
    - **Text Splitter Playground:** https://langchain-text-splitter.streamlit.app/


 <p align="center">
  <img src="../images/RAGe.png" alt="Chunking" style="width: 85%;">
</p>

# Example 1: `CharacterTextSplitter`
- Simple splitting strategy: Splits text based on a single character separator (like space, newline, or comma)
- Fixed separator: Uses only ONE separator throughout the entire text
- Character-based measurement: Measures chunk size by counting characters
- Best for: Simple, uniform text where you know the exact separator to use
- Limitations:
    - Not intelligent about preserving context or semantic meaning
    - May split in the middle of sentences if using space as separator
    - Cannot try multiple separators hierarchically
- Use case: Quick splitting of structured text with consistent formatting

In [2]:
from langchain_text_splitters import CharacterTextSplitter          # Splits text on a single separator by character count

char_splitter = CharacterTextSplitter(
                                    separator=" ",        # Single separator to split on | default: "\n\n" (double newline / paragraph break)
                                    chunk_size=300,       # Max characters per chunk     | default: 1000
                                    chunk_overlap=20,     # Overlapping chars between consecutive chunks to preserve context | default: 200
                                    length_function=len   # Function used to measure chunk size | default: len (character count); can swap to token-counting function
                                    )
char_chunks = char_splitter.split_text(text)              # Split plain string → returns List[str] | alternative: split_documents() for List[Document]


print(f"Original text length: {len(text)} characters\n")            # Total character count of the input text before splitting
print(f"Number of chunks created: {len(char_chunks)}\n")            # Total number of chunks produced after splitting
# Print each chunk with its index, content, and character length
for i, chunk in enumerate(char_chunks, 1):     # enumerate(start=1) → 1-based chunk numbering instead of 0-based
    print(f"Chunk {i}:")
    print(f"'{chunk}'")
    print(f"Length: {len(chunk)} characters")  # Actual character count of this chunk (may be < chunk_size due to separator boundaries)
    print("-" * 50)                            # Visual divider between chunks

Original text length: 1692 characters

Number of chunks created: 6

Chunk 1:
'Dr. Muhammad Arif Butt is an accomplished Assistant Professor at the Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. He holds an MSc and MPhil (both with Gold Medals) and a Ph.D. in Computer Science from PUCIT, University of the Punjab. His research focuses on fuzzy'
Length: 295 characters
--------------------------------------------------
Chunk 2:
'focuses on fuzzy inference models applied to operating systems, embedded systems, and cloud-based services, particularly in decision-making under uncertain and imprecise conditions.

With over 33 years of experience in teaching and management, Dr. Butt has served in both the Pakistan Army and'
Length: 293 characters
--------------------------------------------------
Chunk 3:
'Pakistan Army and University of the Punjab, bringing a wealth of interdisciplinary expertise. His teaching specializations include embedded and real-time operatin

# Example 2: `TokenTextSplitter`
- Token-based measurement: Counts tokens (not characters) using a tokenizer (like tiktoken for OpenAI)
- LLM-compatible: Ensures chunks fit within model token limits (e.g., 8k, 16k tokens)
- Accurate sizing: More precise for LLM context windows since models count tokens, not characters
- Best for: When you need exact control over token counts for API calls
- Limitations:
    - Requires a tokenizer library (like tiktoken)
    - Slightly slower than character-based splitters due to tokenization
    - Still splits mechanically, not semantically
- Use case: Preparing text for specific LLMs with known token limits (GPT-4, Claude, etc.)

In [3]:
from langchain_text_splitters import TokenTextSplitter         # Splits text by token count using OpenAI's tiktoken tokenizer — more accurate than character splitting for LLM token limits

token_splitter = TokenTextSplitter(
                                chunk_size=50,                # Max tokens per chunk (NOT characters) | default: 4000 | 1 token ≈ 4 chars in English (e.g. "hello" = 1 token, "ChatGPT" = 2 tokens)
                                chunk_overlap=10,             # Overlapping tokens between consecutive chunks to preserve context | default: 200
                                encoding_name="cl100k_base",  # tiktoken encoding to use for tokenization | default: "gpt2" | "cl100k_base" is used by GPT-4, GPT-3.5-turbo, text-embedding-ada-002
                                # model_name=None,            # Alternative to encoding_name: auto-selects encoding from model name e.g. "gpt-4" | default: None
                                # allowed_special=set(),      # Set of special tokens (e.g. "<|endoftext|>") allowed during encoding | default: set() i.e. none allowed
                                # disallowed_special="all",   # Special tokens that raise an error if encountered | default: "all" (blocks all special tokens)
                                )

token_chunks = token_splitter.split_text(text)    # Split plain string → returns List[str] | each chunk guaranteed to be ≤ chunk_size tokens (unlike CharacterTextSplitter which approximates)

print(f"Original text length: {len(text)} characters\n")      # Character count of raw input (not token count — use tiktoken to get actual token count)
print(f"Number of chunks created: {len(token_chunks)}\n")     # Total chunks after token-based splitting
for i, chunk in enumerate(token_chunks, 1):       # enumerate(start=1) → 1-based chunk numbering instead of 0-based
    print(f"Chunk {i}:")
    print(f"'{chunk}'")
    print(f"Length: {len(chunk)} characters")     # Shows character length of chunk — to see token count use: len(tokenizer.encode(chunk))
    print("-" * 50)                               # Visual divider between chunks

Original text length: 1692 characters

Number of chunks created: 8

Chunk 1:
'Dr. Muhammad Arif Butt is an accomplished Assistant Professor at the Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. He holds an MSc and MPhil (both with Gold Medals) and a Ph.D. in'
Length: 210 characters
--------------------------------------------------
Chunk 2:
' Gold Medals) and a Ph.D. in Computer Science from PUCIT, University of the Punjab. His research focuses on fuzzy inference models applied to operating systems, embedded systems, and cloud-based services, particularly in decision-making under uncertain and im'
Length: 259 characters
--------------------------------------------------
Chunk 3:
' services, particularly in decision-making under uncertain and imprecise conditions.

With over 33 years of experience in teaching and management, Dr. Butt has served in both the Pakistan Army and University of the Punjab, bringing a wealth of interdisciplinary expertise. His'
Len

# Example 3: `RecursiveCharacterTextSplitter`
- Smart hierarchical splitting: Tries multiple separators in order of preference
- Default separator hierarchy: `["\n\n", "\n", " ", ""]` (paragraph → line → word → character)
- Context preservation: Attempts to keep semantically related text together
- Recursive approach: If a chunk is too large, it recursively splits using the next separator
- Best for: General-purpose text splitting, especially for natural language documents
- Advantages:
    - Preserves paragraph and sentence boundaries when possible
    - More intelligent than simple character splitting
    - Default choice for most RAG applications
- Use case: Books, articles, documentation, general text documents
```mermaid
graph TD
    A["Original Text<br/>1689 characters<br/>4 paragraphs"] --> B{"Try separator: '\n\n'<br/>(paragraph breaks)"}
    
    B --> C["Paragraph 1<br/>425 chars<br/>❌ TOO LARGE > 400"]
    B --> D["Paragraph 2<br/>279 chars<br/>✅ VALID"]
    B --> E["Paragraph 3<br/>567 chars<br/>❌ TOO LARGE > 400"]
    B --> F["Paragraph 4<br/>318 chars<br/>✅ VALID"]
    
    C --> G{"Recursively split P1<br/>Try separator: '\n'<br/>(line breaks)"}
    G --> G1["P1 - Part 1<br/>425 chars<br/>❌ STILL TOO LARGE"]
    
    G1 --> H{"Recursively split<br/>Try separator: ' '<br/>(spaces)"}
    H --> H1["Chunk 1<br/>~400 chars<br/>✅ VALID"]
    H --> H2["Chunk 2 (with overlap)<br/>~45 chars<br/>✅ VALID"]
    
    E --> I{"Recursively split P3<br/>Try separator: '\n'<br/>(line breaks)"}
    I --> I1["P3 - Part 1<br/>567 chars<br/>❌ STILL TOO LARGE"]
    
    I1 --> J{"Recursively split<br/>Try separator: ' '<br/>(spaces)"}
    J --> J1["Chunk 4<br/>~400 chars<br/>✅ VALID"]
    J --> J2["Chunk 5 (with overlap)<br/>~187 chars<br/>✅ VALID"]
    
    H1 --> FINAL["FINAL OUTPUT:<br/>6 Chunks"]
    H2 --> FINAL
    D --> |"Chunk 3"| FINAL
    J1 --> FINAL
    J2 --> FINAL
    F --> |"Chunk 6"| FINAL
    
    style A fill:#e1f5ff
    style B fill:#fff3cd
    style G fill:#fff3cd
    style H fill:#fff3cd
    style I fill:#fff3cd
    style J fill:#fff3cd
    style C fill:#ffcccc
    style E fill:#ffcccc
    style G1 fill:#ffcccc
    style I1 fill:#ffcccc
    style D fill:#ccffcc
    style F fill:#ccffcc
    style H1 fill:#ccffcc
    style H2 fill:#ccffcc
    style J1 fill:#ccffcc
    style J2 fill:#ccffcc
    style FINAL fill:#d4edda
```

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter    # Splits text by trying a hierarchy of separators in order — moves to next separator only if chunks are still too large

recursive_splitter = RecursiveCharacterTextSplitter(
                                                    separators=["\n\n", "\n", " ", ""],  # Ordered list of separators to try | default: ["\n\n", "\n", " ", ""]
                                                                                                 # "\n\n" → split at paragraph breaks first (largest natural boundary)
                                                                                                 # "\n"   → if still too large, split at line breaks
                                                                                                 # " "    → if still too large, split at word boundaries
                                                                                                 # ""     → last resort: split at every character (guarantees chunk_size is never exceeded)
                                                    chunk_size=400,               # Max characters per chunk | default: 1000 | comment says 100 but value is 400 — fix comment to match value
                                                    chunk_overlap=20,             # Overlapping characters between consecutive chunks to preserve context across boundaries | default: 200
                                                    length_function=len,          # Function used to measure chunk size | default: len (character count) | swap to tiktoken for token-based measurement
                                                    # is_separator_regex=False,   # If True, each separator in the list is treated as a regex pattern | default: False
                                                    # keep_separator=False,       # If True, the matched separator is kept at the start of each chunk instead of being discarded | default: False
                                                    # add_start_index=False,      # If True, adds start_index key to chunk metadata indicating its position in original text | default: False
                                                    # strip_whitespace=True,      # If True, strips leading/trailing whitespace from each chunk after splitting | default: True
                                                    )

recursive_chunks = recursive_splitter.split_text(text)    # Split plain string → returns List[str]. Internally: tries "\n\n" first → if any chunk > chunk_size, re-splits that chunk with "\n" → repeats down the separator list
                                                           

print(f"Original text length: {len(text)} characters\n")       # Total character count of input before splitting
print(f"Number of chunks created: {len(recursive_chunks)}\n")  # Total chunks produced — will vary based on text structure
for i, chunk in enumerate(recursive_chunks, 1):    # enumerate(start=1) → 1-based chunk numbering instead of 0-based
    print(f"Chunk {i}:")
    print(f"'{chunk}'")
    print(f"Length: {len(chunk)} characters")      # Actual character count — should always be ≤ chunk_size (unlike CharacterTextSplitter which may exceed it)
    print("-" * 50)                                # Visual divider between chunks

Original text length: 1692 characters

Number of chunks created: 6

Chunk 1:
'Dr. Muhammad Arif Butt is an accomplished Assistant Professor at the Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. He holds an MSc and MPhil (both with Gold Medals) and a Ph.D. in Computer Science from PUCIT, University of the Punjab. His research focuses on fuzzy inference models applied to operating systems, embedded systems, and cloud-based services, particularly'
Length: 399 characters
--------------------------------------------------
Chunk 2:
'particularly in decision-making under uncertain and imprecise conditions.'
Length: 73 characters
--------------------------------------------------
Chunk 3:
'With over 33 years of experience in teaching and management, Dr. Butt has served in both the Pakistan Army and University of the Punjab, bringing a wealth of interdisciplinary expertise. His teaching specializations include embedded and real-time operating systems, system progra

# Example 4: Code-Aware Text Splitting using `RecursiveCharacterTextSplitter` with Language enum
- Syntax-aware splitting: Uses language-specific tokenization rules (e.g., functions, classes, imports) to intelligently divide source code.
- Default separator hierarchy (Python):
```python
["\nclass ", "\ndef ", "\n    ", "\n", " ", ""] # Prioritizes splitting by class, then function, then indentation, then lines, then characters.
```
- Code structure preservation: Tries to keep syntactically related blocks (like a full function or class definition) together.
- Recursive fallback: If a code chunk still exceeds chunk_size, it recursively applies finer-grained separators.
- Best for: Programming source files (Python, JavaScript, etc.) where structural boundaries matter.
- Advantages:
    - Maintains logical units (functions, methods, classes)
    - Avoids cutting code in the middle of syntax blocks
    - Produces cleaner, self-contained chunks suitable for embedding or LLM reasoning
- Use case: Code documentation generation, code search, retrieval-augmented generation (RAG) for programming tasks, code summarization, or LLM fine-tuning on code corpora.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language  # Language enum provides pre-built separator lists for 15+ programming languages

# ── Sample Python code to be split ───────────────────────────────────────────
code = """
class Calculator:                          # Class definition — top-level separator for Python
    def add(self, a, b):                   # Method definition — second-level separator
        return a + b
        
    def multiply(self, a, b):
        return a * b
        
def greet(name):                           # Module-level function — top-level separator
    return f"Hello, {name}!"
    
def main():
    calc = Calculator()
    result = calc.add(5, 3)
    print(f"Result: {result}")
    print(greet("World"))
"""

# ── Initialize code-aware splitter ───────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter.from_language(
                                                        language=Language.PYTHON,   # Language.JS, Language.JAVA, Language.GO, Language.CPP, Language.C, Language.RUBY,  Language.SWIFT, Language.MARKDOWN, Language.LATEX, Language.HTML, Language.SOL             
                                                        chunk_size=100,             # Max characters per chunk | default: 1000 | keep small for code to avoid splitting mid-function
                                                        chunk_overlap=0,            # Overlapping characters between chunks | default: 200 | 0 = no overlap (safe for code since functions are self-contained)
                                                        # length_function=len,      # Function to measure chunk size | default: len (character count)
                                                        # keep_separator=True,      # Whether to keep the separator (e.g. "def ", "class ") at the start of each chunk | default: True for from_language()
                                                        # add_start_index=False,    # If True, adds start_index to chunk metadata showing byte offset in original code | default: False
                                                        # strip_whitespace=False,   # Whether to strip leading/trailing whitespace | default: False for code (indentation is meaningful!)
                                                        )
chunks = splitter.split_text(code)    # Split code string → returns List[str]. Tries Python separators in order — keeps class/def blocks together wherever possible

print(f"\nTotal Chunks: {len(chunks)}")    # Number of code chunks produced
print("=" * 40)
for i, chunk in enumerate(chunks):        # enumerate(start=0) → 0-based index; +1 below for human-readable display
    print(f"\nChunk {i+1}:")
    print(f"Length: {len(chunk)} chars")  # Character count of this chunk — should always be ≤ chunk_size
    print("-" * 20)
    print(repr(chunk))                    # repr() → shows raw string with escape chars (\n, \t) visible — useful for debugging whitespace and indentation
    print("-" * 20)
    print(chunk)                          # Plain print → shows formatted/rendered code with actual newlines and indentation


Total Chunks: 8

Chunk 1:
Length: 94 chars
--------------------
'class Calculator:                          # Class definition — top-level separator for Python'
--------------------
class Calculator:                          # Class definition — top-level separator for Python

Chunk 2:
Length: 83 chars
--------------------
'def add(self, a, b):                   # Method definition — second-level separator'
--------------------
def add(self, a, b):                   # Method definition — second-level separator

Chunk 3:
Length: 12 chars
--------------------
'return a + b'
--------------------
return a + b

Chunk 4:
Length: 46 chars
--------------------
'def multiply(self, a, b):\n        return a * b'
--------------------
def multiply(self, a, b):
        return a * b

Chunk 5:
Length: 88 chars
--------------------
'def greet(name):                           # Module-level function — top-level separator'
--------------------
def greet(name):                           # Module-level fu

# Example 5: `SemanticChunker`
### [Semantic Text Splitting Method Development for RAG](https://github.com/arifpucit/Generative-and-Agentic-AI/blob/main/Research%20Articles/37-Semantic_text_splitting_method_development_for_rag.pdf)
- Meaning-based splitting: Uses embeddings to group semantically similar sentences together and splits when the topic or meaning changes.
- Intelligent boundaries: Splits where topic or meaning changes significantly
- No fixed size: Chunk sizes vary based on semantic similarity, not character/token counts
- Embedding-powered: Uses sentence embeddings to calculate semantic similarity between sentences
- Best for: Documents where preserving topical coherence is critical
- **How it works?**
    - **Break into sentences**: Splits text into individual sentences
    - **Create embeddings**: Converts each sentence into a vector representation
    - **Measure similarity**: Calculates how semantically similar adjacent sentences are
    - **Apply threshold method** to determine "low similarity" cutoff
    - **Split when similarity drops** below the calculated threshold
    - **Group sentences** above threshold into same chunk
- **Advantages:**
    - Most intelligent splitting method
    - Keeps related ideas together even across multiple sentences
    - Better retrieval quality in RAG systems
- **Limitations:**
    - Slower processing and is computationally expensive
    - Requires embedding model to generate embeddings
    - Needs an embedding model
    - Need threshold tuning to adjust for different content types
    - Chunks may vary significantly in size
- Use case: Academic papers, technical documentation, complex narratives where context matters

#### `breakpoint_threshold_type` determines HOW to calculate when to split chunks
- **`"percentile"` (Most Common)**
    - Uses percentage ranking of similarity scores
    - `50` means split when similarity is below 50th percentile
    - Best for: General purpose splitting
- **`"standard_deviation"`**
    - Uses statistical standard deviation from mean similarity
    - `1.0` means split when similarity is 1 standard deviation below average
    - Best for: Documents with consistent writing style
- **`"interquartile"`**
    - Uses interquartile range (IQR) statistical method
    - `1.5` means split at 1.5 × IQR below median
    - Best for: Handling outliers in similarity scores

#### `breakpoint_threshold_amount` sets the sensitivity of splitting
- **For Percentile Type:**
    - **`25`**: Very sensitive - splits often (smaller chunks)
    - **`50`**: Balanced - moderate splitting
    - **`75`**: Less sensitive - splits rarely (larger chunks)
- **For Standard Deviation:**
    - **`0.5`**: Split at half standard deviation below mean
    - **`1.0`**: Split at one standard deviation below mean
    - **`2.0`**: Split at two standard deviations below mean

#### Practical Examples
- **High Sensitivity (More Chunks)**
```python
# Creates many small chunks
breakpoint_threshold_type="percentile"
breakpoint_threshold_amount=25
```
- **Balanced Approach**
```python
# Moderate number of chunks
breakpoint_threshold_type="percentile" 
breakpoint_threshold_amount=50
```
- **Low Sensitivity (Fewer Chunks)**
```python
# Creates fewer, larger chunks
breakpoint_threshold_type="percentile"
breakpoint_threshold_amount=75
```

```
$ uv add langchain-experimental langchain-huggingface sentence-transformers
```

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings          # LangChain wrapper around HuggingFace sentence-transformer models used to convert text into dense vector embeddings
from langchain_experimental.text_splitter import SemanticChunker # Splits text based on semantic meaning rather than fixed size; uses embedding similarity to detect topic shifts

# Load HuggingFace embedding model using LangChain's wrapper
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")   # Sentence-Transformers model used to generate embeddings | lightweight (384-dim vectors) | fast and commonly used for RAG pipelines
                                
# Initialize the SemanticChunker
semantic_splitter = SemanticChunker(
                                    embeddings=embeddings,                  # Embedding model used to compute vector representations of sentences. The splitter compares semantic similarity between adjacent sentences
                                    breakpoint_threshold_type="percentile", # Method used to detect semantic breakpoints between sentences
                                                                                  # "percentile" → split where similarity drops below a percentile threshold
                                                                                  # "standard_deviation" → split when similarity deviation exceeds statistical threshold
                                                                                  # "interquartile" → split based on IQR (robust to outliers)
                                    breakpoint_threshold_amount=50         # Threshold value used with the selected method
                                                                                 # For "percentile": break where similarity is below the 50th percentile (median)
                                                                                 # Lower value → fewer splits (larger chunks)
                                                                                 # Higher value → more splits (smaller chunks)
                                    )
# Perform semantic text splitting
semantic_chunks = semantic_splitter.split_text(text)   # Input: plain string
                                                       # Process:
                                                       # 1. Split text into sentences
                                                       # 2. Convert each sentence into embeddings
                                                       # 3. Compute cosine similarity between adjacent sentence embeddings
                                                       # 4. Detect semantic breakpoints based on threshold rule
                                                       # 5. Group sentences into chunks where semantic continuity exists
                                                       # Output: List[str] where each chunk contains semantically related sentences


# Display chunk statistics
print(f"Number of chunks: {len(semantic_chunks)}\n")    # Total number of semantic chunks created (depends on topic changes in the text)


# Display each chunk
for i, chunk in enumerate(semantic_chunks, 1):          # enumerate(start=1) → chunks numbered starting from 1 instead of 0
    print(f"Chunk {i} (Length: {len(chunk)} chars):")   # Print chunk index and character length
    print(f"'{chunk}'")                                 # Print chunk content
    print("-" * 80)                                     # Visual separator between chunks for readability

Number of chunks: 7

Chunk 1 (Length: 207 chars):
'Dr. Muhammad Arif Butt is an accomplished Assistant Professor at the Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. He holds an MSc and MPhil (both with Gold Medals) and a Ph.D.'
--------------------------------------------------------------------------------
Chunk 2 (Length: 317 chars):
'in Computer Science from PUCIT, University of the Punjab. His research focuses on fuzzy inference models applied to operating systems, embedded systems, and cloud-based services, particularly in decision-making under uncertain and imprecise conditions. With over 33 years of experience in teaching and management, Dr.'
--------------------------------------------------------------------------------
Chunk 3 (Length: 121 chars):
'Butt has served in both the Pakistan Army and University of the Punjab, bringing a wealth of interdisciplinary expertise.'
----------------------------------------------------------------------------

# Example 6: `AgenticChunker`

* **LLM-driven splitting**: Uses a Large Language Model (agent) to decide how to break text into meaningful chunks
* **Context-aware boundaries**: Splits based on understanding of context, intent, and structure—not just similarity
* **Flexible size**: Chunk sizes vary depending on how the agent interprets logical sections
* **Reasoning-powered**: Uses LLM reasoning to decide where one idea ends and another begins
* **Best for**: Complex documents where structure, intent, or meaning is subtle and hard to capture with rules


### How it works?
* **Read full context**: The agent processes the text (or a portion of it)
* **Understand meaning**: Uses LLM reasoning to interpret topics, intent, and flow
* **Decide boundaries**: Dynamically determines where chunks should start and end
* **Generate chunks**: Splits text into logically coherent sections based on understanding

### Advantages:
* Most flexible and human-like chunking method
* Can capture nuanced topic shifts and implicit structure
* Works well even when semantic similarity methods fail
* No need for manual threshold tuning

### Limitations:
* Slower and more expensive (LLM calls required)
* Results may vary depending on the model and prompt
* Less predictable compared to rule-based methods
* Requires careful prompt design for consistency

### Use case:
* Legal documents
* Research papers with complex argument flow
* Multi-topic conversational transcripts
* Any content where **human-like understanding** is required for splitting

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os                       
from dotenv import load_dotenv  


load_dotenv('../keys/.env', override=True) 
openai_api_key = os.getenv('OPENAI_API_KEY')
llm    = ChatOpenAI(model="gpt-4o-mini",   api_key=openai_api_key)   

# Prompt to guide chunking
prompt = ChatPromptTemplate.from_template("""
Split the following text into meaningful chunks.

Rules:
- Each chunk should represent a single coherent idea
- Do not break ideas in the middle
- Keep chunks reasonably sized (3–6 sentences if possible)

Return the output as a numbered list.

TEXT:
{text}
""")

def agentic_chunk(text):
    chain = prompt | llm
    response = chain.invoke({"text": text})
    
    # Simple parsing (assuming numbered list output)
    chunks = []
    for line in response.content.split("\n"):
        if line.strip() and line[0].isdigit():
            chunks.append(line.split(".", 1)[1].strip())
    
    return chunks

# Run chunking
agentic_chunks = agentic_chunk(text)

# Display results
print(f"Number of chunks: {len(agentic_chunks)}\n")
for i, chunk in enumerate(agentic_chunks, 1):
    print(f"Chunk {i} (Length: {len(chunk)} chars):")
    print(f"'{chunk}'")
    print("-" * 80)

Number of chunks: 5

Chunk 1 (Length: 460 chars):
'Dr. Muhammad Arif Butt is an accomplished Assistant Professor at the Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. He holds an MSc and MPhil (both with Gold Medals) and a Ph.D. in Computer Science from PUCIT, University of the Punjab. His research focuses on fuzzy inference models applied to operating systems, embedded systems, and cloud-based services, particularly in decision-making under uncertain and imprecise conditions.'
--------------------------------------------------------------------------------
Chunk 2 (Length: 329 chars):
'With over 33 years of experience in teaching and management, Dr. Butt has served in both the Pakistan Army and University of the Punjab, bringing a wealth of interdisciplinary expertise. His teaching specializations include embedded and real-time operating systems, system programming, cybersecurity, and artificial intelligence.'
---------------------------------------------